<div style="text-align: justify; max-width: 90ch">

# Lecture 03b: An LLM judge and comparing variants

**Status: Mandatory reading.**

`lec_03a` graded answers with code: exact, cheap, and blind to meaning. This notebook lets a model do the grading, an **LLM judge** ([MLflow's judges and scorers](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/)),
on your own machine, and then compares two variants of the RAG system with the judge in the suite.

**What this notebook covers**

- What a keyword cannot read.
- Your own judge with `make_judge`: rubric, temperature, thinking, address.
- Two variants of the system on the same questions and scorers, compared in code and in the UI.

</div>

In [1]:
import os
from pathlib import Path
from typing import Literal

import mlflow
import pandas as pd
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from mlflow.genai.judges import make_judge
from mlflow.genai.scorers import scorer

from course_helpers.checks import check_mlflow
from course_helpers.corpus import load_corpus, load_eval_set
from course_helpers.evaluation import (
    DETERMINISTIC_SCORERS,
    FACTS_RUBRIC,
    compare_variants,
    fact_keywords_found,
    per_question_table,
    build_eval_records_from,
)
from course_helpers.rag import (
    LECTURE_KINDS,
    build_chroma_index,
    make_rag_chain,
    split_documents,
)

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # runs from a lecture subfolder
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()  # no download bars in saved outputs (works after the imports)
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-03-eval"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from course_helpers.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

**Rebuild inputs.** The system and the records of `lec_03a`, from the `course_helpers` helpers:

- the Chroma index of `lec_02d` (reused if it exists, otherwise built now, which takes minutes);
- four chunks per question from the lecture notes and guides (the filter of `lec_02d`), and the chat model with `num_predict=200`;
- five questions of the evaluation set as MLflow records (`build_eval_records_from`), among them q19 for the hand-written answers.

The full set of 19 questions is an exercise: Exercise 3.4.

</div>

In [3]:
check_ollama(OLLAMA_HOST, [EMBED_MODEL])

chunks = split_documents(load_corpus(REPO / "corpus" / "uoa_py_course"))
store = build_chroma_index(
    chunks,
    "course_notes",
    DATA_DIR / "chroma",
    embed_model=EMBED_MODEL,
    ollama_host=OLLAMA_HOST,
)
retriever = store.as_retriever(
    search_kwargs={"k": 4, "filter": {"kind": {"$in": LECTURE_KINDS}}}
)

llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,
    temperature=0,
    num_predict=200,
    # no num_ctx: a judge and a chat model at different windows make Ollama reload the model
)
rag_chain = make_rag_chain(retriever, llm)

DEMO_IDS = ["q04", "q11", "q13", "q15", "q19"]
eval_set = load_eval_set(REPO / "corpus" / "eval" / "qa_eval_set.jsonl")
demo_set = [q for q in eval_set if q["id"] in DEMO_IDS]
records = build_eval_records_from(demo_set)
q19_record = records[DEMO_IDS.index("q19")]

print(f"{len(records)} records; {q19_record['inputs']['question']}")

Ollama OK at http://localhost:11434; models ready: nomic-embed-text


Reusing 'course_notes': 2452 chunks in chroma/
5 records; What is the difference between a list and a tuple?


In [4]:
check_mlflow(MLFLOW_URI)
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
mlflow.tracing.disable_notebook_display()  # keep the trace viewer out of the saved notebook
mlflow.langchain.autolog()

MLflow OK at http://127.0.0.1:5010


<div style="text-align: justify; max-width: 90ch">

## 1. A keyword cannot read

Two answers to q19 of `lec_03a` you can grade yourself: one right in other words, one with the facts reversed. The expected facts are "Tuples are immutable." and "Lists are mutable." The keyword scorer,
now in `course_helpers.evaluation`, grades both:

</div>

In [5]:
HAND_ANSWERS = {
    "right: other words": "A tuple cannot be changed after it is created; a list can.",
    "wrong: facts reversed": "Lists are immutable and tuples are mutable.",
}

pd.Series(
    {
        label: fact_keywords_found(
            outputs=text, expectations=q19_record["expectations"]
        )
        for label, text in HAND_ANSWERS.items()
    }
)

right: other words       0.0
wrong: facts reversed    1.0
dtype: float64

<div style="text-align: justify; max-width: 90ch">

Backwards, on every run: the wrong answer gets 1.0 and the right one 0.0. Telling them apart takes a reader who knows which adjective belongs to which noun.
The idea of an **LLM judge**: give the answer and grading instructions to a model and let it grade.

| Term | Meaning |
| --- | --- |
| **LLM judge** | a scorer that sends the answer and grading instructions to a model and reads back its verdict |
| **rubric** | the grading instructions |
| **verdict** | the judge's value, here `yes` or `no` |
| **rationale** | the judge's explanation, logged next to the verdict |
| **judge model** | the model that grades; here the local chat model that also answers |

Grading with the model that answered is a shortcut: the judge shares its blind spots. `lec_03c` adds a second judge.

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Your own judge with `make_judge`

`make_judge` ([docs](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/make-judge/)) builds a judge from your rubric.
The rubric names what it reads with **template variables**, `{{ inputs }}`, `{{ outputs }}` and `{{ expectations }}`, which MLflow fills from each record (double braces are MLflow's placeholders,
not Python's). The model is a URI: `"ollama:/qwen3:1.7b"` means provider Ollama, model `qwen3:1.7b`.

- **`feedback_value_type`**: `Literal["yes", "no"]` (from Python's `typing`) means "exactly one of these values"; MLflow turns it into the structured output of `lec_01c`.
- **`inference_params`**: `temperature` 0, and `reasoning_effort: "none"`, the `think=False` of `lec_01a` in the OpenAI format.
- **`base_url`**: MLflow calls Ollama itself, without [LangChain](https://github.com/langchain-ai/langchain), at its **OpenAI-compatible endpoint** ([Ollama docs](https://docs.ollama.com/api/openai-compatibility)),
  which speaks the request format of OpenAI's API. MLflow sends each request to `base_url` as given, so it is the full chat address, `/v1/chat/completions`;
  it is built from `OLLAMA_HOST`, so the judge follows your `.env`.

</div>

In [6]:
print(FACTS_RUBRIC)

states_facts = make_judge(
    name="states_facts",
    instructions=FACTS_RUBRIC,
    model=f"ollama:/{CHAT_MODEL}",
    feedback_value_type=Literal["yes", "no"],
    inference_params={"temperature": 0, "reasoning_effort": "none"},
    base_url=f"{OLLAMA_HOST}/v1/chat/completions",
)

Question: {{ inputs }}

Answer: {{ outputs }}

Expected facts: {{ expectations }}

Reply yes if the answer states every expected fact, in any wording, and contradicts none of them. Reply no if any expected fact is missing or the answer states its opposite.


<div style="text-align: justify; max-width: 90ch">

MLflow also ships ready-made judges, such as `Correctness` and `Guidelines` ([built-in judges](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/predefined/)).
On `ollama:/` they use only the default local port `http://localhost:11434`, while `make_judge` takes `base_url` and `inference_params`:
this course uses `make_judge`.

A judge, like a scorer, is called on one answer. It returns a **Feedback**: the verdict and the rationale.

</div>

In [7]:
for label, text in HAND_ANSWERS.items():
    feedback = states_facts(
        inputs=q19_record["inputs"],
        outputs=text,
        expectations=q19_record["expectations"],
    )
    print(f"{label}: {feedback.value}\n  {feedback.rationale}\n")

right: other words: yes
  The answer correctly states the expected facts from the provided source. The facts 'Tuples are immutable.' and 'Lists are mutable.' are both accurately reflected in the answer, and no contradictions are present.



wrong: facts reversed: yes
  The answer correctly states the expected facts from the provided source. The expected facts are: 'Tuples are immutable.' and 'Lists are mutable.' The answer matches these facts exactly, and no contradictions are present.



<div style="text-align: justify; max-width: 90ch">

Read both rationales: does the judge reason about the two facts, and does the reversed answer get `no`? A judge is a model, so its verdict is one sample.
A small judge can get the reversed answer wrong, and a judge at temperature 0 that is wrong is wrong in the same way every time.

**A judge can vary.** Ask once more on the reversed answer, the one a small judge gets wrong more often in example runs:

</div>

In [8]:
feedback = states_facts(
    inputs=q19_record["inputs"],
    outputs=HAND_ANSWERS["wrong: facts reversed"],
    expectations=q19_record["expectations"],
)

print(f"second verdict on the reversed answer: {feedback.value}")

second verdict on the reversed answer: yes


<div style="text-align: justify; max-width: 90ch">

Compare it with the first verdict. At temperature 0 the model takes its most likely token, so a second call within one session usually repeats the verdict.
That is not a guarantee across sessions, machines or Ollama versions: a small judge can sit on the edge between two verdicts.
How often it agrees with a careful reader is a measurement of its own, `lec_03c`.

### ⏸ Pause point

Would you accept a unit test whose result can change on another machine? What would you need before trusting this judge's mean?

</div>

<div style="text-align: justify; max-width: 90ch">

## 3. Two variants on the same questions

A **variant** is one version of the system. Compare variants the way the Python course compared models: same questions, same scorers, one evaluation run each.

- **`k=4`**: the retriever of `lec_02d`, four chunks per question.
- **`k=2`**: two chunks: a shorter prompt, so less to read and pay for, but less chance that the right file is in it.

Each variant is a `predict_fn` with the parameter `question` (`lec_03a`):

</div>

In [9]:
retriever_k2 = store.as_retriever(
    search_kwargs={"k": 2, "filter": {"kind": {"$in": LECTURE_KINDS}}}
)
rag_chain_k2 = make_rag_chain(retriever_k2, llm)


def predict_k4(question):
    """Variant k=4: the RAG chain of lec_02d."""
    return rag_chain.invoke(question)


def predict_k2(question):
    """Variant k=2: the same chain with two chunks per question."""
    return rag_chain_k2.invoke(question)

<div style="text-align: justify; max-width: 90ch">

The trace records how many tokens the chat model read, chunks included: a scorer that returns that number measures the cost side.
The suite is the deterministic scorers of `lec_03a`, this one, and the judge. `make_judge` logs no summary metric unless asked:
`aggregations = ["mean"]` adds the share of `yes`.

</div>

In [10]:
@scorer(aggregations=["mean"])
def prompt_tokens(trace):
    """Number of tokens the model read for this answer: the prompt with its chunks."""
    usage = trace.info.token_usage or {}
    return usage.get("input_tokens", 0)


states_facts.aggregations = ["mean"]
SCORERS = [*DETERMINISTIC_SCORERS, prompt_tokens, states_facts]
results_by_variant = {}

<div style="text-align: justify; max-width: 90ch">

`with mlflow.start_run(run_name=...)` (from `lec_13c` of the Python course) opens a named run; `evaluate` logs into it, so the UI shows the variant's name.
One evaluation per cell: five answers and five verdicts each, plus MLflow's test call on the first question (`lec_03a`).

</div>

In [11]:
with mlflow.start_run(run_name="k=4"):
    results_by_variant["k=4"] = mlflow.genai.evaluate(
        data=records, predict_fn=predict_k4, scorers=SCORERS
    )

2026/10/12 19:05:59 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


2026/10/12 19:05:59 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


In [12]:
with mlflow.start_run(run_name="k=2"):
    results_by_variant["k=2"] = mlflow.genai.evaluate(
        data=records, predict_fn=predict_k2, scorers=SCORERS
    )

2026/10/12 19:06:12 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


<div style="text-align: justify; max-width: 90ch">

`compare_variants` (in `course_helpers.evaluation`) puts both side by side: one column per variant, every metric, plus the mean time per answer (`execution_duration` of each row,
converted to seconds).

</div>

In [13]:
compare_variants(results_by_variant).round(2)

,k=4,k=2
answer_tokens/max,200.00,200.00
answer_tokens/mean,110.60,138.80
cites_expected_source/mean,0.40,0.60
fact_keywords_found/mean,0.70,0.80
prompt_tokens/mean,714.40,389.60
refused/mean,0.00,0.00
retrieval_hit/mean,1.00,0.80
seconds_per_answer/mean,3.82,4.59
states_facts/mean,0.80,0.80


<div style="text-align: justify; max-width: 90ch">

Read the table row by row, as a trade-off:

- **`retrieval_hit`**: did the right file reach the prompt? The retriever is deterministic for a given index, so on your machine this row repeats from run to run.
- **`states_facts`** against **`fact_keywords_found`**: do the judge and the keywords see the same difference?
- **`cites_expected_source`** and **`refused`**: does the answer name its file, or give up?
- **`answer_tokens`**: the tokens the model wrote. A max equal to `num_predict` (200) means at least one answer was cut off, perhaps before its source line.
- **`prompt_tokens`**: the price in tokens. `k=2` sends two chunks instead of four, so its prompt is shorter on every run;
  a hosted model bills each of these tokens.
- **`seconds_per_answer`**: the price in time. It includes waiting, because MLflow sends several questions at once and Ollama answers them in turn:
  compare the columns, not the values.

Five questions are a demonstration, not a decision: one answer moves a mean by 0.2.

</div>

<div style="text-align: justify; max-width: 90ch">

Means say how much, not where. `per_question_table`, the table of `lec_03a` section 6 indexed by question id, for both variants side by side:

</div>

In [14]:
k4 = per_question_table(results_by_variant["k=4"], demo_set, SCORERS)
k2 = per_question_table(results_by_variant["k=2"], demo_set, SCORERS)

pd.DataFrame(
    {
        "hit_k4": k4["retrieval_hit"],
        "hit_k2": k2["retrieval_hit"],
        "judge_k4": k4["states_facts"],
        "judge_k2": k2["states_facts"],
    }
)

,hit_k4,hit_k2,judge_k4,judge_k2
id,,,,
q04,True,True,yes,yes
q11,True,False,no,no
q13,True,True,yes,yes
q15,True,True,yes,yes
q19,True,True,yes,yes


<div style="text-align: justify; max-width: 90ch">

Find q11 (`weights='distance'`): if `hit_k4` is True and `hit_k2` False, its file ranked third or fourth, so only `k=4` gave it to the model.
Did the judge's verdict on q11 follow the retrieval? Without its file the model can only refuse or answer from memory, and a judge that reads only the question,
the answer and the facts cannot tell which one happened. Read the q11 answers in the UI.

### ⏸ Pause point

In the MLflow UI, open experiment `llm-course-03-eval`: the two runs are named after their variant. Select both and compare them.
Which variant would you ship, and what would you tell the person who pays for the tokens?

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- A keyword scorer cannot read: an **LLM judge** gives the answer and a **rubric** to a model and logs a **verdict** with its **rationale**.
- `make_judge` takes your rubric, `inference_params` (temperature 0, thinking off) and `base_url` (your `OLLAMA_HOST`).
- A judge is a model: it can be wrong, the same way every time, and its verdicts can change between sessions. `lec_03c` measures how far to trust it.
- **Compare variants** on the same questions and scorers, one named run each; read the metrics as a trade-off, and the rows for where it changed.

The helpers live in `course_helpers.evaluation`: `FACTS_RUBRIC`, `make_facts_judge`, `prompt_tokens` and `compare_variants`.

**Next: `lec_03c`**, which checks the judge itself: its verdicts against labels you trust, and what it can see.

</div>